# Data Transform

In [1]:
# import libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim

In [2]:
# define paths to parquet files

names_countries = "../data/silver/df_join_fn_cc_cleaned/"


In [3]:
# set a spark session

def create_spark_session(app_name:str):
    spark = SparkSession.builder \
        .appName(app_name) \
        .master("local[*]") \
        .getOrCreate()
    return spark

In [4]:
spark = create_spark_session("gold_pipeline")

In [5]:
df_names_country = spark.read.parquet(names_countries)
df_names_country.show(5)

+--------+------+-------+-----+------------+------------+
|forename|gender|country|count|country_code|country_name|
+--------+------+-------+-----+------------+------------+
|   Zeret|     M|     PS|    9|          PS|   palestine|
|    عآزف|     M|     PS|    9|          PS|   palestine|
|  Mhmood|     F|     PS|    9|          PS|   palestine|
|   خليفه|     M|     PS|    9|          PS|   palestine|
|   סטיין|     M|     PS|    9|          PS|   palestine|
+--------+------+-------+-----+------------+------------+
only showing top 5 rows


## 1. Adapting the cleaned data for Deep Learning model

In [6]:
# Define a list with LATAM country names, to seek it on country_naem dataframe
"""
In this spet I ensure that the resulting name match with LATAM names
"""
latam_countries = [
    "argentina", "bolivia", "brazil", "chile", "colombia", "costa rica",
    "cuba", "ecuador", "el salvador", "guatemala", "honduras", "mexico",
    "nicaragua", "panama", "paraguay", "peru", "puerto rico", "dominican republic",
    "uruguay", "venezuela"
]

df_latam = df_names_country.filter(col('country_name').rlike("|".join(latam_countries)))
df_latam.show()

+----------------+------+-------+-----+------------+------------+
|        forename|gender|country|count|country_code|country_name|
+----------------+------+-------+-----+------------+------------+
|      Mynor Ivan|     M|     GT|    4|          GT|   guatemala|
|  Leonel Antonio|     M|     GT|    4|          GT|   guatemala|
|          Anparo|     F|     GT|    4|          GT|   guatemala|
|           Henio|     M|     GT|    4|          GT|   guatemala|
|         Medelyn|     F|     GT|    4|          GT|   guatemala|
|           Matia|     M|     GT|    4|          GT|   guatemala|
| Mynor Francisco|     M|     GT|    4|          GT|   guatemala|
|           Jersi|     M|     GT|    4|          GT|   guatemala|
|          Adaliz|     F|     GT|    4|          GT|   guatemala|
|           Jeili|     F|     GT|    4|          GT|   guatemala|
|Lobito Solitario|     M|     GT|    4|          GT|   guatemala|
| Leonel Fernando|     M|     GT|    4|          GT|   guatemala|
|         

In [7]:
# Count the number of records on latam name dataframe
"""
As a result we got 1,447,779 records with LATAM names
"""
df_latam.count()

1447779

In [16]:
# Resume how many records has every country code
df_latam.groupBy('country_name').count().sort(col("count").desc()).show()

+------------+------+
|country_name| count|
+------------+------+
|    colombia|331917|
|      mexico|230800|
|        peru|223930|
|      brazil|188422|
|       chile|151531|
|     bolivia| 75105|
|   argentina| 55951|
|      panama| 54875|
|   guatemala| 45977|
|  costa rica| 38041|
|     uruguay| 31612|
|     ecuador| 11559|
| puerto rico|  6373|
|    honduras|  1240|
| el salvador|   446|
+------------+------+



### a. Remove unnecessary columns and change columns name for neural network training

In [9]:
# Removils unnecessary columns

df_latam_nn = df_latam.drop("country", "count", "country_code", "country_name")
df_latam_nn.show()

+----------------+------+
|        forename|gender|
+----------------+------+
|      Mynor Ivan|     M|
|  Leonel Antonio|     M|
|          Anparo|     F|
|           Henio|     M|
|         Medelyn|     F|
|           Matia|     M|
| Mynor Francisco|     M|
|           Jersi|     M|
|          Adaliz|     F|
|           Jeili|     F|
|Lobito Solitario|     M|
| Leonel Fernando|     M|
|          Jeilin|     F|
|         Adalina|     F|
|           Jeily|     F|
|             Ęel|     M|
|          Jerome|     M|
|           Matio|     M|
|            Jero|     F|
|          Lastor|     M|
+----------------+------+
only showing top 20 rows


In [10]:
df_latam_nn = df_latam_nn.withColumnRenamed("forename", "NAME")
df_latam_nn = df_latam_nn.withColumnRenamed("gender", "GENDER")
df_latam_nn.show()


+----------------+------+
|            NAME|GENDER|
+----------------+------+
|      Mynor Ivan|     M|
|  Leonel Antonio|     M|
|          Anparo|     F|
|           Henio|     M|
|         Medelyn|     F|
|           Matia|     M|
| Mynor Francisco|     M|
|           Jersi|     M|
|          Adaliz|     F|
|           Jeili|     F|
|Lobito Solitario|     M|
| Leonel Fernando|     M|
|          Jeilin|     F|
|         Adalina|     F|
|           Jeily|     F|
|             Ęel|     M|
|          Jerome|     M|
|           Matio|     M|
|            Jero|     F|
|          Lastor|     M|
+----------------+------+
only showing top 20 rows


In [11]:
def count_null_values(df):
    for c in df.columns:
        print(c, ": ", df.filter(col(c).isNull()).count())

In [12]:
count_null_values(df_latam_nn)

NAME :  0
GENDER :  0


In [14]:
df_latam_nn.printSchema()

root
 |-- NAME: string (nullable = true)
 |-- GENDER: string (nullable = true)



### b. Save dataframe adapted to neural network

In [13]:
df_latam_nn.write.mode("overwrite").parquet("../data/gold/df_latam_names_for_nn_training/")